In [1]:
import glob
import math
import os
from pathlib import Path
import random

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from shapely.geometry import box, Polygon, Point
from shapely.ops import triangulate
from tqdm import tqdm
import rasterio
import shutil

WORK_DIR = Path('/Users/muhammadabdul/Desktop/penn/earth_genome_internship/climate_trace/data')

In [2]:
def uniform_random_points_in_polygon(polygon, num_points, neg_polygon=None):
    triangles = triangulate(polygon)
    areas = np.array([tri.area for tri in triangles])
    probs = areas / areas.sum()

    points = []
    while len(points) < num_points:
        tri = np.random.choice(triangles, p=probs)
        
        a, b, c = tri.exterior.coords[:3]
        r1 = np.random.rand()
        r2 = np.random.rand()
        sqrt_r1 = np.sqrt(r1)
        x = (1 - sqrt_r1) * a[0] + (sqrt_r1 * (1 - r2)) * b[0] + (sqrt_r1 * r2) * c[0]
        y = (1 - sqrt_r1) * a[1] + (sqrt_r1 * (1 - r2)) * b[1] + (sqrt_r1 * r2) * c[1]
        
        point = Point(x, y)
        
        if neg_polygon and neg_polygon.contains(point):
            continue  
    
        if polygon.contains(point):
            points.append(point)
            


    return gpd.GeoDataFrame(geometry=points, crs="EPSG:4326")

# Sample  lot

In [3]:
# load spatial prior and labels, filter label to only include areas within the spatial prior
sp_path = WORK_DIR / 'KS_barton_lot_pond' / 'spatial_prior.geojson'
sp_polys = gpd.read_file(sp_path)
sp_polys["geometry"] = sp_polys.geometry.envelope
sp_polys = gpd.GeoDataFrame(geometry=sp_polys.buffer(0.001), crs='epsg:4326')

label_path = WORK_DIR / 'KS_barton_lot_pond' / 'lot.geojson'
labels = gpd.read_file(label_path)
labels = gpd.overlay(labels, sp_polys, how='intersection')

/var/folders/zf/m0p4r__d7gb0cphf85xj8z180000gn/T/ipykernel_61124/1665147800.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  sp_polys = gpd.GeoDataFrame(geometry=sp_polys.buffer(0.001), crs='epsg:4326')


In [4]:
# compute area of each polygon and buffer to avoid sampling near the boundary
utm = 'epsg:32614'
labels = gpd.GeoDataFrame(geometry=labels.buffer(-0.0001), crs='epsg:4326')
labels['Area'] = labels.to_crs(utm).geometry.area/1e4

/var/folders/zf/m0p4r__d7gb0cphf85xj8z180000gn/T/ipykernel_61124/2352091395.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  labels = gpd.GeoDataFrame(geometry=labels.buffer(-0.0001), crs='epsg:4326')


In [5]:
# sample points from each polygon proportional to its area
total_points_sample = 5000

gdf = gpd.GeoDataFrame()
for i, row in tqdm(labels.iterrows()):
    num_points=math.ceil((row.Area / labels['Area'].sum()) * total_points_sample)
    pts = uniform_random_points_in_polygon(row.geometry, num_points)
    gdf = gpd.pd.concat([gdf, pts])

37it [00:00, 99.28it/s] 


In [6]:
# save points to geojson
outpath = WORK_DIR / 'KS_barton_lot_pond'  / f"lot_sample_points_sp.geojson"
gdf.to_file(outpath, index=False)

# Sample pond

In [7]:
# load spatial prior and labels, filter label to only include areas within the spatial prior
sp_path = WORK_DIR / 'KS_barton_lot_pond' / 'spatial_prior.geojson'
sp_polys = gpd.read_file(sp_path)
sp_polys["geometry"] = sp_polys.geometry.envelope
sp_polys = gpd.GeoDataFrame(geometry=sp_polys.buffer(0.001), crs='epsg:4326')

label_path = WORK_DIR / 'KS_barton_lot_pond' / 'pond.geojson'
labels = gpd.read_file(label_path)
labels = gpd.overlay(labels, sp_polys, how='intersection')

/var/folders/zf/m0p4r__d7gb0cphf85xj8z180000gn/T/ipykernel_61124/1991776533.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  sp_polys = gpd.GeoDataFrame(geometry=sp_polys.buffer(0.001), crs='epsg:4326')


In [8]:
# compute area of each polygon and buffer to avoid sampling near the boundary
utm = 'epsg:32614'
labels = gpd.GeoDataFrame(geometry=labels.buffer(-0.0001), crs='epsg:4326')
labels['Area'] = labels.to_crs(utm).geometry.area/1e4

/var/folders/zf/m0p4r__d7gb0cphf85xj8z180000gn/T/ipykernel_61124/2352091395.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  labels = gpd.GeoDataFrame(geometry=labels.buffer(-0.0001), crs='epsg:4326')


In [9]:
# sample points from each polygon proportional to its area
total_points_sample = 5000

gdf = gpd.GeoDataFrame()
for i, row in tqdm(labels.iterrows()):
    num_points=math.ceil((row.Area / labels['Area'].sum()) * total_points_sample)
    pts = uniform_random_points_in_polygon(row.geometry, num_points)
    gdf = gpd.pd.concat([gdf, pts])

45it [00:00, 195.55it/s]


In [10]:
# save points to geojson
outpath = WORK_DIR / 'KS_barton_lot_pond'  / f"pond_sample_points_sp.geojson"
gdf.to_file(outpath, index=False)

# Sample other

In [ ]:
# load county polygon
county_path = WORK_DIR / 'KS_barton_lot_pond' / 'county_poly.geojson'
county_poly = gpd.read_file(county_path)

# load spatial prior
sp_path = WORK_DIR / 'KS_barton_lot_pond' / 'spatial_prior.geojson'
sp_poly = gpd.read_file(sp_path)

# convert polys to bounding boxes to deacrese chance of sampling points in region of interest
sp_poly["geometry"] = sp_poly.geometry.envelope
sp_poly = gpd.GeoDataFrame(geometry=sp_poly.buffer(0.001), crs='epsg:4326')

/var/folders/zf/m0p4r__d7gb0cphf85xj8z180000gn/T/ipykernel_61449/2415883154.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  sp_poly = gpd.GeoDataFrame(geometry=sp_poly.buffer(0.001), crs='epsg:4326')


In [4]:
# load all tifs in county
tif_dir = WORK_DIR / "KS_barton_lot_pond" / "county_polyAlphaEarth"
tif_paths = list(tif_dir.rglob("*.tif")) + list(tif_dir.rglob("*.tiff"))

# create union of spatial prior polygons to speed up intersection checks
sp_union = sp_poly.union_all()
intersecting_tifs = []
extent_polygons = []

# check intersection of each tif with spatial prior union and store intersecting tifs and their extent polygons
for tif_path in tqdm(tif_paths):
    with rasterio.open(tif_path) as src:
        if src.crs is None:
            continue
        
        # create polygon from tif bounds and reproject to spatial prior CRS
        bounds_poly = box(src.bounds.left, src.bounds.bottom, src.bounds.right, src.bounds.top)
        bounds_poly = gpd.GeoSeries([bounds_poly], crs=src.crs).to_crs(sp_poly.crs).iloc[0]

        # check intersection with spatial prior union and store results
        if bounds_poly.intersects(sp_union):
            intersecting_tifs.append(tif_path)
            extent_polygons.append(bounds_poly)
            
# create geodataframe of intersecting tif extents and merge to create single polygon for sampling
extent_gdf = gpd.GeoDataFrame({"tif_path": intersecting_tifs}, geometry=extent_polygons, crs=sp_poly.crs)
merged_extent_polygon = extent_gdf.union_all()
merged_extent_gdf = gpd.GeoDataFrame(geometry=[merged_extent_polygon], crs=sp_poly.crs)

100%|██████████| 400/400 [04:06<00:00,  1.63it/s] 


In [5]:
# copy intersecting tifs to new directory for sampling
dst_dir = WORK_DIR / "KS_barton_lot_pond" / "county_polyAlphaEarth_sp"
dst_dir.mkdir(parents=True, exist_ok=True)

# copy each intersecting tif to new directory, preserving relative paths
for src_path in intersecting_tifs:
    rel_path = src_path.relative_to(tif_dir)
    target_path = dst_dir / rel_path
    target_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_path, target_path)

In [6]:
# clip merged extent polygon to county boundary to avoid sampling outside of county
clipped_extent_gdf = gpd.overlay(
    merged_extent_gdf,
    county_poly[["geometry"]],
    how="intersection",
)

In [7]:
# subtract spatial prior union from merged extent to avoid sampling points in spatial prior polygons, then compute area for sampling
clipped_extent_gdf = clipped_extent_gdf.difference(sp_poly.union_all())
utm = 'epsg:32614'
clipped_extent_gdf = gpd.GeoDataFrame(geometry=clipped_extent_gdf, crs='epsg:4326').dropna()
clipped_extent_gdf['Area'] = clipped_extent_gdf.to_crs(utm).geometry.area/1e4

In [8]:
# sample points from clipped extent proportional to area, avoiding spatial prior polygons
gdf = gpd.GeoDataFrame()
total_points_sample = 7000
for i, row in tqdm(clipped_extent_gdf.iterrows()):
    num_points=math.ceil((row.Area / clipped_extent_gdf['Area'].sum()) * total_points_sample)
    pts = uniform_random_points_in_polygon(row.geometry, num_points)
    gdf = gpd.pd.concat([gdf, pts])

1it [00:10, 10.04s/it]


In [9]:
# save points to geojson
outpath = WORK_DIR / 'KS_barton_lot_pond' / f"other_sample_points_sp.geojson"
gdf.to_file(outpath, index=False)